
# Advanced Sequential LangGraph Workflow (Self-Correcting)

This notebook demonstrates a sophisticated **Sequential Workflow** featuring nested conditional feedback loops. It is designed to emulate a multi-stage editorial process:

1. **Outline Generation**: Creates an outline based on a topic.
2. **Outline Review**: A specialized agent reviews the outline. If it's poor, it sends it back with feedback.
3. **Blog Drafting**: Once the outline is approved, it drafts the full article.
4. **Blog Evaluation**: An editor reviews the full draft. If it fails quality checks, it is sent back for revision.

**Key Concepts Demonstrated:**
- Sequential Node Execution
- Multiple Conditional Edges (Routing)
- Cyclic Graphs (Loops)
- State persistence and counting (Max Revisions)


In [ ]:

import os
from typing import TypedDict
from IPython.display import display, Image
import textwrap

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# --- 1. SETUP ---

# Initialize LLM. We use a lower temperature for the reviewers and a slightly higher one for the writer.
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
llm_strict = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

print("✅ LLMs initialized.")


In [ ]:

# --- 2. STATE DEFINITION ---

# Define the State schema that all nodes will pass around
class BlogState(TypedDict):
    topic: str
    
    # Outline phase
    outline: str
    outline_feedback: str
    outline_revisions: int
    outline_approved: bool
    
    # Draft phase
    blog: str
    blog_feedback: str
    blog_revisions: int
    blog_approved: bool

print("✅ BlogState schema defined.")


In [ ]:

# --- 3. DEFINE NODES ---

def generate_outline_node(state: BlogState):
    print("-> \033[94mGenerating/Revising Outline...\033[0m")
    topic = state['topic']
    feedback = state.get('outline_feedback', '')
    revisions = state.get('outline_revisions', 0)
    
    prompt = f"Generate a detailed, logical outline for a blog post about: {topic}.\n"
    if feedback:
        prompt += f"\nCRITICAL PREVIOUS FEEDBACK: You must fix these issues:\n{feedback}\n"
        prompt += f"\nPREVIOUS OUTLINE:\n{state.get('outline', '')}\n"
        
    response = llm_creative.invoke([SystemMessage(content="You are an expert content strategist."), HumanMessage(content=prompt)])
    
    return {"outline": response.content, "outline_revisions": revisions + 1}


def review_outline_node(state: BlogState):
    print("-> \033[93mReviewing Outline...\033[0m")
    outline = state['outline']
    topic = state['topic']
    
    prompt = f"Review this outline for a blog post about '{topic}'.\n"
    prompt += "Does it logically cover the topic? Are there any glaring omissions?\n"
    prompt += "If it is excellent, reply with 'APPROVED'.\n"
    prompt += "If it needs work, reply with 'REJECTED: ' followed by detailed feedback on what needs to be changed.\n\n"
    prompt += f"OUTLINE:\n{outline}"
    
    response = llm_strict.invoke([SystemMessage(content="You are a strict editorial director."), HumanMessage(content=prompt)])
    content = response.content.strip()
    
    if content.startswith("APPROVED"):
        print("   [OUTLINE STATUS: \033[92mAPPROVED\033[0m]")
        return {"outline_approved": True, "outline_feedback": ""}
    else:
        feedback = content.replace("REJECTED:", "").strip()
        print("   [OUTLINE STATUS: \033[91mREJECTED\033[0m]")
        return {"outline_approved": False, "outline_feedback": feedback}


def generate_blog_node(state: BlogState):
    print("-> \033[92mDrafting Blog Content...\033[0m")
    topic = state['topic']
    outline = state['outline']
    feedback = state.get('blog_feedback', '')
    revisions = state.get('blog_revisions', 0)
    
    prompt = f"Write a comprehensive and engaging blog post about '{topic}'.\n"
    prompt += f"You MUST strictly follow this approved outline:\n{outline}\n\n"
    
    if feedback:
        prompt += f"\nCRITICAL EDITORIAL FEEDBACK TO ADDRESS:\n{feedback}\n"
        prompt += f"\nPREVIOUS DRAFT:\n{state.get('blog', '')}\n"
        
    response = llm_creative.invoke([SystemMessage(content="You are a senior technical writer."), HumanMessage(content=prompt)])
    
    return {"blog": response.content, "blog_revisions": revisions + 1}


def evaluate_blog_node(state: BlogState):
    print("-> \033[95mEvaluating Final Draft...\033[0m")
    outline = state['outline']
    blog = state['blog']
    
    prompt = "Evaluate this blog post against its outline.\n"
    prompt += "Does it deliver on the outline's promises? Is the pacing good? Is the tone appropriate?\n"
    prompt += "If it is ready to publish, reply with 'PUBLISH'.\n"
    prompt += "If it needs revisions, reply with 'REVISE: ' followed by specific actionable feedback.\n\n"
    prompt += f"OUTLINE:\n{outline}\n\n"
    prompt += f"DRAFT:\n{blog}\n"
    
    response = llm_strict.invoke([SystemMessage(content="You are the Editor-in-Chief making the final call."), HumanMessage(content=prompt)])
    content = response.content.strip()
    
    if content.startswith("PUBLISH"):
        print("   [BLOG STATUS: \033[92mPUBLISH READY\033[0m]")
        return {"blog_approved": True, "blog_feedback": ""}
    else:
        feedback = content.replace("REVISE:", "").strip()
        print("   [BLOG STATUS: \033[91mNEEDS REVISION\033[0m]")
        return {"blog_approved": False, "blog_feedback": feedback}

print("✅ All Nodes defined.")


In [ ]:

# --- 4. CONDITIONAL LOGIC AND GRAPH BUILDER ---

# Conditional Routers

def check_outline_status(state: BlogState):
    if state.get("outline_approved"):
        return "generate_blog"
    elif state.get("outline_revisions", 0) >= 3:
        print("-> Max outline revisions reached. Forcing progression.")
        return "generate_blog"
    else:
        return "generate_outline"


def check_blog_status(state: BlogState):
    if state.get("blog_approved"):
        return "END"
    elif state.get("blog_revisions", 0) >= 3:
        print("-> Max blog revisions reached. Forcing completion.")
        return "END"
    else:
        return "generate_blog"


# Build Graph Structure
builder = StateGraph(BlogState)

# Add Nodes
builder.add_node("generate_outline", generate_outline_node)
builder.add_node("review_outline", review_outline_node)
builder.add_node("generate_blog", generate_blog_node)
builder.add_node("evaluate_blog", evaluate_blog_node)


# Add Edges (The Flow)

# 1. Start by making an outline
builder.add_edge(START, "generate_outline")

# 2. Outline always goes to review
builder.add_edge("generate_outline", "review_outline")

# 3. Review conditionally routes back to outline, OR moves forward to blog generation
builder.add_conditional_edges(
    "review_outline",
    check_outline_status,
    {
        "generate_outline": "generate_outline", # Loop back
        "generate_blog": "generate_blog"        # Move forward
    }
)

# 4. Blog generation always goes to evaluation
builder.add_edge("generate_blog", "evaluate_blog")

# 5. Evaluation conditionally routes back to generation, OR finishes
builder.add_conditional_edges(
    "evaluate_blog",
    check_blog_status,
    {
        "generate_blog": "generate_blog", # Loop back
        "END": END                        # Finish
    }
)

# Compile
graph = builder.compile()

print("✅ Advanced Sequential Graph built and compiled.")

# (Optional) Display Graph Image
try:
    png_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
except Exception as e:
    print("Could not display graph map:", e)


In [ ]:

# --- 5. RUN THE WORKFLOW ---

initial_state = {
    "topic": "The Psychological Impact of Infinite Scrolling on Human Attention Spans",
    "outline_revisions": 0,
    "blog_revisions": 0,
    "outline_approved": False,
    "blog_approved": False
}

print(f"Starting Advanced Pipeline for topic: '{initial_state['topic']}'\n")

# Run the graph
final_state = graph.invoke(initial_state)

print("\n" + "="*50)
print("\033[1mFINAL ARTICLE PUBLISHED:\033[0m")
print("="*50 + "\n")

content_to_save = final_state.get('blog', 'No blog generated.')
print(content_to_save)


In [ ]:

# --- 6. SAVE TO FILE ---
output_filename = "infinite_scroll_impact.txt" 

try:
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {initial_state['topic']}\n")
        f.write(f"OUTLINE REVISIONS: {final_state.get('outline_revisions', 0)}\n")
        f.write(f"BLOG REVISIONS: {final_state.get('blog_revisions', 0)}\n")
        f.write("="*50 + "\n\n")
        
        # Word wrap the text file output
        wrapped_content = textwrap.fill(content_to_save, width=80)
        f.write(wrapped_content)
        
    print(f"\n✅ Successfully saved the result to '{output_filename}'.")
except Exception as e:
    print(f"\n❌ Failed to save the file: {e}")
